# RUAP LV1 — Dohvaćanje podataka putem REST API-ja i mrežnih stranica

Rješenja zadataka za samostalan rad iz odjeljka 5. Bilježnica redom rješava tri zadatka:
HTTP zahtjeve prema JSONPlaceholderu, povijesne vremenske podatke za Osijek, Zagreb i Split
te izdvajanje citata s prve dvije stranice Quotes to Scrape.

Izvorni odgovori spremaju se u `RUAP_LV1_podaci/izvorno`, a izdvojeni zapisi u
`RUAP_LV1_podaci/obradeno`. Ćelije treba pokretati odozgo prema dolje.


In [1]:
%pip install -q requests beautifulsoup4 pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Priprema

Radni direktoriji i funkcija za spremanje izvornog odgovora preuzeti su iz odjeljaka 1.2 i 3.3.
Uz tijelo odgovora sprema se i mala datoteka s vremenom dohvata i točnom adresom zahtjeva.


In [2]:
import json
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

RAD = Path("RUAP_LV1_podaci")
IZVORNO = RAD / "izvorno"
OBRADENO = RAD / "obradeno"
IZVORNO.mkdir(parents=True, exist_ok=True)
OBRADENO.mkdir(parents=True, exist_ok=True)

JSONPLACEHOLDER = "https://jsonplaceholder.typicode.com"
ZAGLAVLJA = {"Accept": "application/json"}


def spremi_odgovor(odgovor, putanja, preuzeto_utc):
    meta = {
        "retrieved_at_utc": preuzeto_utc,
        "request_url": odgovor.url,
    }
    putanja.write_bytes(odgovor.content)
    meta_putanja = putanja.with_suffix(".meta.json")
    meta_putanja.write_text(
        json.dumps(meta, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


## Zadatak 1. HTTP zahtjevi i odgovori

### 1.a) Zbirka i pojedinačni resurs

Uspoređuju se dva GET zahtjeva: filtrirana zbirka `/todos?userId=1&id=1` i pojedinačni resurs `/todos/1`.


In [3]:
odgovor_zbirka = requests.get(
    f"{JSONPLACEHOLDER}/todos",
    params={"userId": 1, "id": 1},
    headers=ZAGLAVLJA,
    timeout=30,
)
odgovor_resurs = requests.get(
    f"{JSONPLACEHOLDER}/todos/1",
    headers=ZAGLAVLJA,
    timeout=30,
)

for naziv, odgovor in (("zbirka", odgovor_zbirka), ("resurs", odgovor_resurs)):
    odgovor.raise_for_status()
    podaci = odgovor.json()
    print(f"=== {naziv} ===")
    print("URL:", odgovor.url)
    print("status:", odgovor.status_code)
    print("Content-Type:", odgovor.headers.get("Content-Type"))
    print("oblik:", type(podaci).__name__)
    print(json.dumps(podaci, ensure_ascii=False, indent=2))
    print()

print(
    "zapis u listi jednak je objektu resursa:",
    odgovor_zbirka.json()[0] == odgovor_resurs.json(),
)


=== zbirka ===
URL: https://jsonplaceholder.typicode.com/todos?userId=1&id=1
status: 200
Content-Type: application/json; charset=utf-8
oblik: list
[
  {
    "userId": 1,
    "id": 1,
    "title": "delectus aut autem",
    "completed": false
  }
]

=== resurs ===
URL: https://jsonplaceholder.typicode.com/todos/1
status: 200
Content-Type: application/json; charset=utf-8
oblik: dict
{
  "userId": 1,
  "id": 1,
  "title": "delectus aut autem",
  "completed": false
}

zapis u listi jednak je objektu resursa: True


Oba zahtjeva vraćaju status **200** i `Content-Type: application/json`.

Razlika je u obliku tijela. `GET /todos?userId=1&id=1` filtrira zbirku, pa je tijelo JSON lista
s jednim objektom. `GET /todos/1` dohvaća jedan resurs prema identifikatoru, pa je tijelo JSON objekt.
Sadržaj tog objekta i jedinog elementa liste isti je zapis. To je razlika iz odjeljka 2.2: filtrirana
zbirka vraća listu, a pojedinačni resurs objekt.


### 1.b) Nepostojeći zadatak

Zahtjev ide na `/todos/999999`. Iznimku koju podigne `raise_for_status()` treba obraditi, a ne
prekinuti bilježnicu.


In [4]:
odgovor_404 = requests.get(
    f"{JSONPLACEHOLDER}/todos/999999",
    headers=ZAGLAVLJA,
    timeout=30,
)
print("status:", odgovor_404.status_code)
print("Content-Type:", odgovor_404.headers.get("Content-Type"))
print("tijelo kao tekst:", odgovor_404.text)
print("JSON se može pročitati i kod pogreške:", odgovor_404.json())

try:
    odgovor_404.raise_for_status()
    print("Zahtjev je uspio, što ovdje ne očekujemo.")
except requests.HTTPError as pogreska:
    print("uhvaćena iznimka:", type(pogreska).__name__)
    print(pogreska)


status: 404
Content-Type: application/json; charset=utf-8
tijelo kao tekst: {}
JSON se može pročitati i kod pogreške: {}
uhvaćena iznimka: HTTPError
404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/todos/999999


Resurs ne postoji, pa poslužitelj vraća **404 Not Found**. Tijelo je `{}`: to je valjan JSON, ali
nije zapis zadatka. Zato `response.json()` uspije i kad zahtjev nije uspio.

Status se provjerava prije čitanja podataka jer poruka o pogrešci također može biti JSON.
Uspješan parse ne znači da je stigao očekivani resurs. `raise_for_status()` za statuse 4xx i 5xx
podiže `requests.HTTPError`, pa se pogreška vidi prije nego što se tijelo protumači kao zadatak.


### 1.c) Svi zadaci korisnika 2 i samo dovršeni

Parametri idu kroz argument `params`. Za `completed` API očekuje tekst `"true"`.


In [5]:
svi = requests.get(
    f"{JSONPLACEHOLDER}/todos",
    params={"userId": 2},
    headers=ZAGLAVLJA,
    timeout=30,
)
svi.raise_for_status()
svi_podaci = svi.json()

dovrseni = requests.get(
    f"{JSONPLACEHOLDER}/todos",
    params={"userId": 2, "completed": "true"},
    headers=ZAGLAVLJA,
    timeout=30,
)
dovrseni.raise_for_status()
dovrseni_podaci = dovrseni.json()

print("URL svih:", svi.url)
print("URL dovršenih:", dovrseni.url)
print("broj svih zadataka:", len(svi_podaci))
print("broj dovršenih:", len(dovrseni_podaci))
print("svi imaju userId 2:", all(zapis["userId"] == 2 for zapis in svi_podaci))
print("dovršeni imaju userId 2:", all(zapis["userId"] == 2 for zapis in dovrseni_podaci))
print(
    "dovršeni imaju completed true:",
    all(zapis["completed"] is True for zapis in dovrseni_podaci),
)
print(
    "dovršeni su podskup svih:",
    {zapis["id"] for zapis in dovrseni_podaci} <= {zapis["id"] for zapis in svi_podaci},
)


URL svih: https://jsonplaceholder.typicode.com/todos?userId=2
URL dovršenih: https://jsonplaceholder.typicode.com/todos?userId=2&completed=true
broj svih zadataka: 20
broj dovršenih: 8
svi imaju userId 2: True
dovršeni imaju userId 2: True
dovršeni imaju completed true: True
dovršeni su podskup svih: True


Korisnik 2 ima 20 zadataka, a 8 ih je dovršeno. Svaki vraćeni zapis ima `userId` 2. U filtriranom
odgovoru je uz to `completed` jednak `true`, a identifikatori dovršenih zadataka podskup su svih
njegovih zadataka. Biblioteka Requests iz rječnika `params` sama sastavlja upitni niz, uključujući
`completed=true`.


### 1.d) HEAD i GET za isti resurs

`requests.head()` šalje HEAD na `/todos/1`. Usporedba ide po statusu, odabranim zaglavljima i tijelu.


In [6]:
get_odgovor = requests.get(
    f"{JSONPLACEHOLDER}/todos/1",
    headers=ZAGLAVLJA,
    timeout=30,
)
head_odgovor = requests.head(
    f"{JSONPLACEHOLDER}/todos/1",
    headers=ZAGLAVLJA,
    timeout=30,
)
get_odgovor.raise_for_status()
head_odgovor.raise_for_status()


def sazetak(odgovor):
    return {
        "status": odgovor.status_code,
        "Content-Type": odgovor.headers.get("Content-Type"),
        "etag": odgovor.headers.get("etag"),
        "Transfer-Encoding": odgovor.headers.get("Transfer-Encoding"),
        "duljina_tijela": len(odgovor.content),
    }


print("GET:", sazetak(get_odgovor))
print("HEAD:", sazetak(head_odgovor))
print("GET tijelo:", get_odgovor.text)
print("HEAD tijelo:", repr(head_odgovor.text))


GET: {'status': 200, 'Content-Type': 'application/json; charset=utf-8', 'etag': 'W/"53-hfEnumeNh6YirfjyjaujcOPPT+s"', 'Transfer-Encoding': 'chunked', 'duljina_tijela': 83}
HEAD: {'status': 200, 'Content-Type': 'application/json; charset=utf-8', 'etag': 'W/"53-hfEnumeNh6YirfjyjaujcOPPT+s"', 'Transfer-Encoding': None, 'duljina_tijela': 0}
GET tijelo: {
  "userId": 1,
  "id": 1,
  "title": "delectus aut autem",
  "completed": false
}
HEAD tijelo: ''


Status, `Content-Type` i `etag` podudaraju se: oba odgovora opisuju istu reprezentaciju.
Tijelo GET odgovora je JSON zapis zadatka, a tijelo HEAD odgovora je prazno (`duljina_tijela` je 0).

HEAD je ista radnja kao GET, samo bez tijela. Zato GET ima `Transfer-Encoding: chunked`
(tijelo se šalje u dijelovima), a HEAD to zaglavlje nema. `requests.head()` po zadanim
postavkama ne prati preusmjeravanja; ova adresa vraća 200 izravno, pa to na usporedbu ne utječe.


## Zadatak 2. Dohvat vremenskih podataka

### Izvor, uvjeti i opis polja

Izvor je [Open-Meteo Historical Weather API](https://archive-api.open-meteo.com/v1/archive),
model reanalize ERA5. Vrijednosti se odnose na točku prostorne mreže blizu tražene lokacije,
ne na očitanje jedne meteorološke postaje. Vraćene koordinate zato se mogu malo razlikovati
od traženih.

Razdoblje je od 1. do 7. siječnja 2024., uključivo, u UTC-u. Jedan redak tablice je jedan grad
i jedan sat.

| Grad | Geografska širina | Geografska dužina |
| --- | --- | --- |
| Osijek | 45.55 | 18.70 |
| Zagreb | 45.81 | 15.98 |
| Split | 43.51 | 16.44 |

Podaci su dostupni pod licencom [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
Besplatni pristup namijenjen je nekomercijalnoj uporabi, uključujući nastavu. Uz skup treba
navesti izvor, poveznicu na licencu i napravljene promjene.

Promjene u odnosu na puni odgovor API-ja: zadržana su tri grada, navedeni tjedan i odabrane
satne varijable, a liste iz polja `hourly` spojene su u jednu tablicu sa stupcem `location`.

| Polje | Značenje i jedinica |
| --- | --- |
| `location` | Naziv traženog grada. |
| `time_utc` | Vremenski zapis u UTC-u (u odgovoru: `iso8601`). |
| `temperature_2m` | Temperatura zraka na 2 m, °C. |
| `relative_humidity_2m` | Relativna vlažnost na 2 m, %. |
| `precipitation` | Količina oborine u prethodnom satu, mm. |
| `wind_speed_10m` | Brzina vjetra na 10 m, km/h. Samo u dopunjenoj tablici iz zadatka 2.c. |

Jedinice su potvrđene iz polja `hourly_units` u ćeliji zadatka 2.c i zapisane u
`obradeno/opis_vremena.json`.


In [7]:
GRADOVI = (
    ("Osijek", 45.55, 18.70),
    ("Zagreb", 45.81, 15.98),
    ("Split", 43.51, 16.44),
)
API_URL = "https://archive-api.open-meteo.com/v1/archive"
SATNE = ["temperature_2m", "relative_humidity_2m", "precipitation"]


def parametri_vremena(sirina, duzina, varijable):
    return {
        "latitude": sirina,
        "longitude": duzina,
        "start_date": "2024-01-01",
        "end_date": "2024-01-07",
        "hourly": ",".join(varijable),
        "timezone": "UTC",
        "models": "era5",
    }


def dohvati_grad(grad, sirina, duzina, varijable, naziv_datoteke):
    odgovor = requests.get(
        API_URL,
        params=parametri_vremena(sirina, duzina, varijable),
        timeout=30,
    )
    preuzeto = datetime.now(timezone.utc).isoformat()
    odgovor.raise_for_status()
    spremi_odgovor(odgovor, IZVORNO / naziv_datoteke, preuzeto)
    return odgovor


def tablica_iz_hourly(podaci, grad):
    tablica = pd.DataFrame(podaci["hourly"])
    tablica = tablica.rename(columns={"time": "time_utc"})
    tablica["time_utc"] = pd.to_datetime(tablica["time_utc"], utc=True)
    tablica.insert(0, "location", grad)
    return tablica


### 2.a) Tri grada, zaseban izvorni odgovor

Za svaki grad šalje se isti upit: isto razdoblje, iste satne varijable, `timezone=UTC` i `models=era5`.
Tijelo i metapodaci spremaju se u zasebne datoteke.


In [8]:
for grad, sirina, duzina in GRADOVI:
    odgovor = dohvati_grad(grad, sirina, duzina, SATNE, f"{grad.lower()}.json")
    podaci = odgovor.json()
    print(grad)
    print("  status:", odgovor.status_code)
    print("  URL:", odgovor.url)
    print("  tražene koordinate:", sirina, duzina)
    print("  vraćene koordinate:", podaci["latitude"], podaci["longitude"])
    print("  broj sati:", len(podaci["hourly"]["time"]))
    print("  jedinice:", podaci["hourly_units"])


Osijek
  status: 200
  URL: https://archive-api.open-meteo.com/v1/archive?latitude=45.55&longitude=18.7&start_date=2024-01-01&end_date=2024-01-07&hourly=temperature_2m%2Crelative_humidity_2m%2Cprecipitation&timezone=UTC&models=era5
  tražene koordinate: 45.55 18.7
  vraćene koordinate: 45.5 18.75
  broj sati: 168
  jedinice: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation': 'mm'}


Zagreb
  status: 200
  URL: https://archive-api.open-meteo.com/v1/archive?latitude=45.81&longitude=15.98&start_date=2024-01-01&end_date=2024-01-07&hourly=temperature_2m%2Crelative_humidity_2m%2Cprecipitation&timezone=UTC&models=era5
  tražene koordinate: 45.81 15.98
  vraćene koordinate: 45.75 16.0
  broj sati: 168
  jedinice: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation': 'mm'}


Split
  status: 200
  URL: https://archive-api.open-meteo.com/v1/archive?latitude=43.51&longitude=16.44&start_date=2024-01-01&end_date=2024-01-07&hourly=temperature_2m%2Crelative_humidity_2m%2Cprecipitation&timezone=UTC&models=era5
  tražene koordinate: 43.51 16.44
  vraćene koordinate: 43.5 16.5
  broj sati: 168
  jedinice: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation': 'mm'}


### 2.b) Zajednička tablica `vrijeme.csv`

Tablica se slaže iz spremljenih JSON odgovora, bez novog zahtjeva. Stupci su oni iz odjeljka 3.4.
Nakon spremanja datoteka se ponovno učitava, a vremenski stupac se vraća u UTC jer CSV ne čuva tip.


In [9]:
tablice = []
for grad, *_ostatak in GRADOVI:
    podaci = json.loads((IZVORNO / f"{grad.lower()}.json").read_text(encoding="utf-8"))
    tablice.append(tablica_iz_hourly(podaci, grad))

vrijeme = pd.concat(tablice, ignore_index=True)
print("broj redaka:", len(vrijeme))
print("stupci:", vrijeme.columns.tolist())
print(vrijeme.dtypes)
print(vrijeme["location"].value_counts())
print(vrijeme.head(3))

csv_putanja = OBRADENO / "vrijeme.csv"
vrijeme.to_csv(csv_putanja, index=False, encoding="utf-8")

ponovno = pd.read_csv(csv_putanja)
ponovno["time_utc"] = pd.to_datetime(ponovno["time_utc"], utc=True)
print("učitano redaka:", len(ponovno))
print("učitani stupci:", ponovno.columns.tolist())
print("tip time_utc:", ponovno["time_utc"].dtype)
print("podaci i vremenske oznake sačuvani:", vrijeme.equals(ponovno))


broj redaka: 504
stupci: ['location', 'time_utc', 'temperature_2m', 'relative_humidity_2m', 'precipitation']
location                                str
time_utc                datetime64[us, UTC]
temperature_2m                      float64
relative_humidity_2m                  int64
precipitation                       float64
dtype: object
location
Osijek    168
Zagreb    168
Split     168
Name: count, dtype: int64
  location                  time_utc  temperature_2m  relative_humidity_2m  \
0   Osijek 2024-01-01 00:00:00+00:00             8.0                    78   
1   Osijek 2024-01-01 01:00:00+00:00             8.0                    78   
2   Osijek 2024-01-01 02:00:00+00:00             8.0                    78   

   precipitation  
0            0.0  
1            0.0  
2            0.1  
učitano redaka: 504
učitani stupci: ['location', 'time_utc', 'temperature_2m', 'relative_humidity_2m', 'precipitation']
tip time_utc: datetime64[us, UTC]
podaci i vremenske oznake sačuvani: T

Sedam dana po 24 sata daje 168 redaka po gradu, ukupno 504. `time_utc` nakon učitavanja
opet ima zonu UTC, a usporedba s tablicom prije spremanja pokazuje da su vrijednosti sačuvane.


### 2.c) Brzina vjetra

Upit se proširuje varijablom `wind_speed_10m`. Podaci za sva tri grada dohvaćaju se ponovno,
izvorni odgovori spremaju se zasebno (datoteke `*_vjetar.json`), a dopunjena tablica ide u
`vrijeme_vjetar.csv`. Jedinica brzine vjetra čita se iz `hourly_units`.


In [10]:
satne_s_vjetrom = SATNE + ["wind_speed_10m"]
tablice_vjetar = []
jedinice = None

for grad, sirina, duzina in GRADOVI:
    odgovor = dohvati_grad(
        grad, sirina, duzina, satne_s_vjetrom, f"{grad.lower()}_vjetar.json"
    )
    podaci = odgovor.json()
    jedinice = podaci["hourly_units"]
    tablice_vjetar.append(tablica_iz_hourly(podaci, grad))

vrijeme_vjetar = pd.concat(tablice_vjetar, ignore_index=True)
print("hourly_units:", jedinice)
print("stupci:", vrijeme_vjetar.columns.tolist())
print("broj redaka:", len(vrijeme_vjetar))
print(vrijeme_vjetar.head(3))

csv_vjetar = OBRADENO / "vrijeme_vjetar.csv"
vrijeme_vjetar.to_csv(csv_vjetar, index=False, encoding="utf-8")
ponovno_vjetar = pd.read_csv(csv_vjetar)
ponovno_vjetar["time_utc"] = pd.to_datetime(ponovno_vjetar["time_utc"], utc=True)
print("CSV sadrži wind_speed_10m:", "wind_speed_10m" in ponovno_vjetar.columns)
print("dopunjena tablica sačuvana:", vrijeme_vjetar.equals(ponovno_vjetar))

opis_polja = {
    "location": "Naziv traženog grada.",
    "time_utc": f"Vremenski zapis ({jedinice['time']}), zona UTC.",
    "temperature_2m": f"Temperatura zraka na 2 m, {jedinice['temperature_2m']}.",
    "relative_humidity_2m": (
        f"Relativna vlažnost na 2 m, {jedinice['relative_humidity_2m']}."
    ),
    "precipitation": (
        f"Količina oborine u prethodnom satu, {jedinice['precipitation']}."
    ),
    "wind_speed_10m": f"Brzina vjetra na 10 m, {jedinice['wind_speed_10m']}.",
}
(OBRADENO / "opis_vremena.json").write_text(
    json.dumps(
        {
            "izvor": "Open-Meteo Historical Weather API",
            "adresa": API_URL,
            "model": "era5",
            "licenca": "CC BY 4.0",
            "licenca_url": "https://creativecommons.org/licenses/by/4.0/",
            "razdoblje_utc": ["2024-01-01", "2024-01-07"],
            "promjene": (
                "Odabrani su Osijek, Zagreb i Split te satne varijable za tjedan "
                "1.–7. siječnja 2024. Liste iz hourly spojene su u tablicu sa stupcem location."
            ),
            "polja": opis_polja,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("opis polja dopunjen jedinicama iz odgovora:")
for polje, opis in opis_polja.items():
    print(f"  {polje}: {opis}")


hourly_units: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation': 'mm', 'wind_speed_10m': 'km/h'}
stupci: ['location', 'time_utc', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m']
broj redaka: 504
  location                  time_utc  temperature_2m  relative_humidity_2m  \
0   Osijek 2024-01-01 00:00:00+00:00             8.0                    78   
1   Osijek 2024-01-01 01:00:00+00:00             8.0                    78   
2   Osijek 2024-01-01 02:00:00+00:00             8.0                    78   

   precipitation  wind_speed_10m  
0            0.0            12.4  
1            0.0            12.6  
2            0.1            11.6  
CSV sadrži wind_speed_10m: True
dopunjena tablica sačuvana: True
opis polja dopunjen jedinicama iz odgovora:
  location: Naziv traženog grada.
  time_utc: Vremenski zapis (iso8601), zona UTC.
  temperature_2m: Temperatura zraka na 2 m, °C.
  relative_humidity_2m: Relativna vlažnost n

`hourly_units` za brzinu vjetra daje **km/h**. Temperatura je u °C, relativna vlažnost u %,
a oborina u mm. Tim jedinicama dopunjen je opis polja iz odjeljka 3.4; isti opis zapisan je i u
`opis_vremena.json`. Zajednička tablica i dalje ima 504 retka, a novi stupac je `wind_speed_10m`.


### Ponovno izdvajanje vremenskih podataka bez novog dohvata

Sljedeća ćelija čita samo spremljene JSON datoteke s vjetrom i ponovno gradi tablicu.


In [11]:
iz_datoteka = []
for grad, *_ostatak in GRADOVI:
    podaci = json.loads(
        (IZVORNO / f"{grad.lower()}_vjetar.json").read_text(encoding="utf-8")
    )
    iz_datoteka.append(tablica_iz_hourly(podaci, grad))

ponovljena_tablica = pd.concat(iz_datoteka, ignore_index=True)
print("redaka iz spremljenih odgovora:", len(ponovljena_tablica))
print("jednako tablici iz zadatka 2.c:", ponovljena_tablica.equals(vrijeme_vjetar))


redaka iz spremljenih odgovora: 504
jednako tablici iz zadatka 2.c: True


## Zadatak 3. Izdvajanje citata

### Izvor, uvjeti i opis polja

Izvor je [Quotes to Scrape](https://quotes.toscrape.com/), dio projekta Web Scraping Sandbox,
namijenjen učenju izdvajanja podataka iz HTML-a. Na stranici nije navedena licenca tekstova
citata; to je uz skup izričito zabilježeno. `robots.txt` daje upute automatiziranim klijentima,
ali ne određuje licencu. Provjera te datoteke je u sljedećoj ćeliji.

| Polje | Značenje |
| --- | --- |
| `text` | Tekst citata. |
| `author` | Autor. |
| `tags` | Lista oznaka. Oznake ostaju lista jer je skup spremljen kao JSON, ne kao CSV. |
| `source_url` | Adresa stranice s koje je citat izdvojen. |


In [12]:
robots = requests.get("https://quotes.toscrape.com/robots.txt", timeout=30)
print("status:", robots.status_code)
print("Content-Type:", robots.headers.get("Content-Type"))
print(robots.text[:400])


status: 404
Content-Type: text/html; charset=utf-8
<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>



Zahtjev na `/robots.txt` vraća **404 Not Found**. Poslužitelj nema tu datoteku, pa nema ni
posebnih uputa za putanje. Stranica je ogledna i predviđena za vježbu ovog tipa. Licenca
tekstova i dalje nije navedena.


### 3.a) Prve dvije stranice preko poveznice Next

Obilazak počinje na početnoj adresi, sprema HTML i metapodatke svake stranice te relativni
`href` spaja s `urljoin()`. Zaustavlja se nakon dvije stranice ili čim sljedeće poveznice nema.


In [13]:
url = "https://quotes.toscrape.com/"
broj_stranica = 0

while url is not None and broj_stranica < 2:
    odgovor = requests.get(url, timeout=30)
    preuzeto = datetime.now(timezone.utc).isoformat()
    odgovor.raise_for_status()
    broj_stranica += 1
    spremi_odgovor(odgovor, IZVORNO / f"quotes_page_{broj_stranica}.html", preuzeto)

    soup = BeautifulSoup(odgovor.content, "html.parser")
    dalje = soup.select_one("li.next a")
    sljedeci = urljoin(odgovor.url, dalje["href"]) if dalje else None
    print(f"stranica {broj_stranica}: status {odgovor.status_code}")
    print("  URL:", odgovor.url)
    print("  sljedeća poveznica:", sljedeci)
    url = sljedeci

print("spremljeno stranica:", broj_stranica)


stranica 1: status 200
  URL: https://quotes.toscrape.com/
  sljedeća poveznica: https://quotes.toscrape.com/page/2/


stranica 2: status 200
  URL: https://quotes.toscrape.com/page/2/
  sljedeća poveznica: https://quotes.toscrape.com/page/3/
spremljeno stranica: 2


### 3.b) Tekst, autor, oznake i adresa izvora

Zapisi se izdvajaju iz spremljenog HTML-a. Adresa izvora čita se iz metapodataka, dakle iz
zahtjeva kojim je stranica stvarno dohvaćena. Svi zapisi idu u `quotes.json`.


In [14]:
def citati_iz_html(sadrzaj, source_url):
    soup = BeautifulSoup(sadrzaj, "html.parser")
    blokovi = soup.select("div.quote")
    if not blokovi:
        raise ValueError("Na stranici nisu pronađeni citati.")
    zapisi = []
    for blok in blokovi:
        zapisi.append(
            {
                "text": blok.select_one("span.text").get_text(" ", strip=True),
                "author": blok.select_one("small.author").get_text(" ", strip=True),
                "tags": [
                    oznaka.get_text(strip=True) for oznaka in blok.select(".tags a.tag")
                ],
                "source_url": source_url,
            }
        )
    return zapisi


citati = []
for broj in range(1, broj_stranica + 1):
    putanja = IZVORNO / f"quotes_page_{broj}.html"
    meta = json.loads(putanja.with_suffix(".meta.json").read_text(encoding="utf-8"))
    stranica = citati_iz_html(putanja.read_bytes(), meta["request_url"])
    print(f"{putanja.name}: {len(stranica)} citata, izvor {meta['request_url']}")
    print("  autori:", ", ".join(zapis["author"] for zapis in stranica))
    citati.extend(stranica)

(OBRADENO / "quotes.json").write_text(
    json.dumps(citati, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("ukupno citata:", len(citati))
print("prva tri zapisa:")
for zapis in citati[:3]:
    print(json.dumps(zapis, ensure_ascii=False, indent=2))


quotes_page_1.html: 10 citata, izvor https://quotes.toscrape.com/
  autori: Albert Einstein, J.K. Rowling, Albert Einstein, Jane Austen, Marilyn Monroe, Albert Einstein, André Gide, Thomas A. Edison, Eleanor Roosevelt, Steve Martin
quotes_page_2.html: 10 citata, izvor https://quotes.toscrape.com/page/2/
  autori: Marilyn Monroe, J.K. Rowling, Albert Einstein, Bob Marley, Dr. Seuss, Douglas Adams, Elie Wiesel, Friedrich Nietzsche, Mark Twain, Allen Saunders
ukupno citata: 20
prva tri zapisa:
{
  "text": "“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”",
  "author": "Albert Einstein",
  "tags": [
    "change",
    "deep-thoughts",
    "thinking",
    "world"
  ],
  "source_url": "https://quotes.toscrape.com/"
}
{
  "text": "“It is our choices, Harry, that show what we truly are, far more than our abilities.”",
  "author": "J.K. Rowling",
  "tags": [
    "abilities",
    "choices"
  ],
  "source_url": "https://quotes.tosc

Svaka od dvije spremljene stranice ima 10 blokova `div.quote`, ukupno 20 citata. Autori i tekst
u ispisu isti su kao u spremljenom HTML-u jer se zapisi grade iz tih datoteka, a ne iz zasebnog
zahtjeva. Oznake ostaju lista unutar JSON objekta.


### 3.c) Citati s jednom oznakom

Odabrana je oznaka `life`. Pojavljuje se među prikupljenim oznakama, i to na objema stranicama.
U zasebnu datoteku ulaze samo citati koji je imaju.


In [15]:
oznaka = "life"
if oznaka not in {tag for zapis in citati for tag in zapis["tags"]}:
    raise ValueError(f"Oznaka {oznaka!r} nije među prikupljenim citatima.")

odabrani = [zapis for zapis in citati if oznaka in zapis["tags"]]
putanja_oznake = OBRADENO / f"quotes_{oznaka}.json"
putanja_oznake.write_text(
    json.dumps(odabrani, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("oznaka:", oznaka)
print("datoteka:", putanja_oznake.as_posix())
print("broj citata:", len(odabrani))
for zapis in odabrani:
    print(f"- {zapis['author']} | {zapis['source_url']}")
    print(f"  {zapis['text']}")


oznaka: life
datoteka: RUAP_LV1_podaci/obradeno/quotes_life.json
broj citata: 6
- Albert Einstein | https://quotes.toscrape.com/
  “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
- André Gide | https://quotes.toscrape.com/
  “It is better to be hated for what you are than to be loved for what you are not.”
- Marilyn Monroe | https://quotes.toscrape.com/page/2/
  “This life is what you make it. No matter what, you're going to mess up sometimes, it's a universal truth. But the good part is you get to decide how you're going to mess it up. Girls will be your friends - they'll act like it anyway. But just remember, some come, some go. The ones that stay with you through everything - they're your true best friends. Don't let go of them. Also remember, sisters make the best friends in the world. As for lovers, well, they'll come and go too. And baby, I hate to say it, most of them - actually pretty much all o

### Ponovno izdvajanje citata bez novog dohvata

Ćelija ponovno čita samo `quotes_page_1.html` i `quotes_page_2.html` te uspoređuje rezultat
sa spremljenim `quotes.json`.


In [16]:
ponovljeni = []
for broj in (1, 2):
    putanja = IZVORNO / f"quotes_page_{broj}.html"
    meta = json.loads(putanja.with_suffix(".meta.json").read_text(encoding="utf-8"))
    ponovljeni.extend(citati_iz_html(putanja.read_bytes(), meta["request_url"]))

spremljeni = json.loads((OBRADENO / "quotes.json").read_text(encoding="utf-8"))
print("citata iz spremljenog HTML-a:", len(ponovljeni))
print("jednako quotes.json:", ponovljeni == spremljeni)

print("datoteke:")
for putanja in sorted(RAD.rglob("*")):
    if putanja.is_file():
        print(f"  {putanja.as_posix()} ({putanja.stat().st_size} B)")


citata iz spremljenog HTML-a: 20
jednako quotes.json: True
datoteke:
  RUAP_LV1_podaci/izvorno/osijek.json (5589 B)
  RUAP_LV1_podaci/izvorno/osijek.meta.json (287 B)
  RUAP_LV1_podaci/izvorno/osijek_vjetar.json (6400 B)
  RUAP_LV1_podaci/izvorno/osijek_vjetar.meta.json (304 B)
  RUAP_LV1_podaci/izvorno/quotes_page_1.html (11064 B)
  RUAP_LV1_podaci/izvorno/quotes_page_1.meta.json (112 B)
  RUAP_LV1_podaci/izvorno/quotes_page_2.html (13745 B)
  RUAP_LV1_podaci/izvorno/quotes_page_2.meta.json (119 B)
  RUAP_LV1_podaci/izvorno/split.json (5715 B)
  RUAP_LV1_podaci/izvorno/split.meta.json (288 B)
  RUAP_LV1_podaci/izvorno/split_vjetar.json (6550 B)
  RUAP_LV1_podaci/izvorno/split_vjetar.meta.json (305 B)
  RUAP_LV1_podaci/izvorno/zagreb.json (5596 B)
  RUAP_LV1_podaci/izvorno/zagreb.meta.json (288 B)
  RUAP_LV1_podaci/izvorno/zagreb_vjetar.json (6386 B)
  RUAP_LV1_podaci/izvorno/zagreb_vjetar.meta.json (305 B)
  RUAP_LV1_podaci/obradeno/opis_vremena.json (816 B)
  RUAP_LV1_podaci/obradeno

Izdvajanje se može ponoviti iz spremljenih odgovora. Vremenska tablica s vjetrom nastaje iz
`osijek_vjetar.json`, `zagreb_vjetar.json` i `split_vjetar.json`, a citati iz dviju HTML datoteka.
Novi mrežni zahtjev za tu provjeru nije potreban.
